# Baseline BM25 --- Notebook de apoio

**Disciplina:** Inteligência Artificial --- FACOM/UFMS --- 2026/1

Este notebook carrega o `corpus.jsonl` gerado pelo notebook de coleta (`coleta_arxiv.ipynb`), treina um índice **BM25** e devolve o *top-k* para uma consulta-exemplo. É o seu **baseline obrigatório**. A partir daqui, você só precisa adicionar pré-processamento mais cuidadoso, comparar com KNN/denso e implementar os módulos de aprofundamento.

**Atenção:** este código é propositalmente "mínimo viável". Você é avaliado também pelo **rigor das suas escolhas** (justificativa dos hiperparâmetros, pré-processamento, conexão com o paradigma probabilístico de Naïve Bayes). Não entregue isto como está --- evolua-o.

In [11]:
# !pip install rank_bm25 nltk pandas

In [12]:
import json
import re
from pathlib import Path

import pandas as pd
from rank_bm25 import BM25Okapi

# stopwords do NLTK (somente uma vez)
import nltk
try:
    from nltk.corpus import stopwords
    STOP = set(stopwords.words("english"))
except LookupError:
    nltk.download("stopwords")
    from nltk.corpus import stopwords
    STOP = set(stopwords.words("english"))

## 1. Carregamento do corpus

Ajuste o caminho se o seu `corpus.jsonl` estiver em outro lugar.

In [13]:
CORPUS_PATH = Path("./data/corpus.jsonl")  # ajuste se necessário

docs = []
with open(CORPUS_PATH, "r", encoding="utf-8") as f:
    for line in f:
        docs.append(json.loads(line))

print(f"Documentos carregados: {len(docs)}")
docs[0]

Documentos carregados: 2000


{'arxiv_id': '2311.07578',
 'title': 'A Metacognitive Approach to Out-of-Distribution Detection for Segmentation',
 'abstract': "Despite outstanding semantic scene segmentation in closed-worlds, deep neural networks segment novel instances poorly, which is required for autonomous agents acting in an open world. To improve out-of-distribution (OOD) detection for segmentation, we introduce a metacognitive approach in the form of a lightweight module that leverages entropy measures, segmentation predictions, and spatial context to characterize the segmentation model's uncertainty and detect pixel-wise OOD data in real-time. Additionally, our approach incorporates a novel method of generating synthetic OOD data in context with in-distribution data, which we use to fine-tune existing segmentation models with maximum entropy training. This further improves the metacognitive module's performance without requiring access to OOD data while enabling compatibility with established pre-trained mod

## 2. Pré-processamento

Mínimo aceitável: *lower-casing*, remoção de pontuação, remoção de *stopwords*. Não usamos *stemming* aqui para não esconder discussão (você pode adicionar e comparar).

In [14]:
# Remoção de pontuações e aplicação das letras minúsculas.
TOKEN_RE = re.compile(r"[A-Za-z][A-Za-z\-]+")

def tokenize(text: str):
    text = text.lower()
    tokens = TOKEN_RE.findall(text)
    return [t for t in tokens if t not in STOP and len(t) > 2]

tokenize("LayoutLM: Pre-training of text and layout for document image understanding")

['layoutlm',
 'pre-training',
 'text',
 'layout',
 'document',
 'image',
 'understanding']

In [15]:
# Texto indexado = título + abstract
texts = [(d["title"] + ". " + d["abstract"]) for d in docs]
tokenized_corpus = [tokenize(t) for t in texts]
print("Exemplo de tokens do doc 0:", tokenized_corpus[0][:20])

Exemplo de tokens do doc 0: ['metacognitive', 'approach', 'out-of-distribution', 'detection', 'segmentation', 'despite', 'outstanding', 'semantic', 'scene', 'segmentation', 'closed-worlds', 'deep', 'neural', 'networks', 'segment', 'novel', 'instances', 'poorly', 'required', 'autonomous']


## 3. Índice BM25

Hiperparâmetros padrão do `rank_bm25`: $k_1=1.5$, $b=0.75$. Documente no seu relatório qual valor adotou e por quê. O módulo M4 (otimização) pode ajustá-los empiricamente.

In [16]:
# Os parâmetros foram escolhidos de acordo com o padrão da literatura
bm25 = BM25Okapi(tokenized_corpus, k1=1.5, b=0.75)
print("Vocabulário BM25:", len(bm25.idf), "termos")

Vocabulário BM25: 17787 termos


## 4. Função de busca

In [17]:
# Função para atribuir o score do BM25 e retornar em ordem decrescente
def search(query: str, k: int = 10):
    q_tokens = tokenize(query)
    scores = bm25.get_scores(q_tokens)
    top_idx = scores.argsort()[::-1][:k]
    return [(int(i), float(scores[i]), docs[i]) for i in top_idx]

In [18]:
query = "Pre-training of text and layout for document image understanding"
results = search(query, k=10)

for rank, (idx, score, d) in enumerate(results, 1):
    print(f"{rank:>2}. [{score:6.2f}] {d['title']}")
    print(f"     {d['arxiv_id']} | {d.get('primary_category')} | {d.get('published','')[:10]}")
    print(f"     {d['abstract'][:200]}...")
    print()

 1. [ 26.05] LayoutLLM: Large Language Model Instruction Tuning for Visually Rich Document Understanding
     2403.14252 | cs.CL | 2024-03-21
     This paper proposes LayoutLLM, a more flexible document analysis method for understanding imaged documents. Visually Rich Document Understanding tasks, such as document image classification and inform...

 2. [ 23.73] Spatial Information Integration in Small Language Models for Document Layout Generation and Classification
     2501.05497 | cs.CL | 2025-01-09
     Document layout understanding is a field of study that analyzes the spatial arrangement of information in a document hoping to understand its structure and layout. Models such as LayoutLM (and its sub...

 3. [ 20.95] XY-Cut++: Advanced Layout Ordering via Hierarchical Mask Mechanism on a Novel Benchmark
     2504.10258 | cs.CV | 2025-04-14
     Document Reading Order Recovery is a fundamental task in document image understanding, playing a pivotal role in enhancing Retrieval-Augme

## 5. Gerando um arquivo `runs/bm25.trec` para avaliação

Formato TREC tradicional: `qid Q0 doc_id rank score system`. Esse formato é aceito por `pytrec_eval` e `ir_measures`, e é o que o script `evaluate.py` (no diretório `eval/`) consome.

In [19]:
queries_path = Path("./eval/queries.tsv")  # qid<TAB>texto da query
runs_dir = Path("./runs");
runs_dir.mkdir(exist_ok=True)
run_path = runs_dir / "bm25.trec"

# Roda o BM25 utilizando as 10 queries e salva o top 5 resultados
queries = pd.read_csv(queries_path, sep="\t", names=["qid", "text"])
with open(run_path, "w", encoding="utf-8") as f:
    for _, q in queries.iterrows():
        for rank, (idx, score, d) in enumerate(search(q["text"], k=5), 1):
            f.write(f"{q['qid']} Q0 {d['arxiv_id']} {rank} {score:.6f} bm25\n")

print("Run salva em:", run_path)

with open(run_path, "r", encoding="utf-8") as f:
    for i, line in zip(range(5), f):
        print(line.rstrip())

Run salva em: runs\bm25.trec
q1 Q0 2501.05497 1 18.536300 bm25
q1 Q0 2403.14252 2 13.767602 bm25
q1 Q0 2503.04065 3 13.141926 bm25
q1 Q0 2310.14785 4 11.416240 bm25
q1 Q0 2504.04085 5 10.767124 bm25


## 6. Próximos passos

1. Substitua o tokenizador por algo mais robusto (e.g., spaCy) e compare.
2. Implemente um segundo *retriever* (KNN + TF-IDF, ou KNN + *embeddings*) e gere `runs/knn.trec`.
3. Construa as `qrels` (Seção 2 do template em `eval/qrels_example.tsv`).
4. Use `eval/evaluate.py` para comparar.
5. Implemente o(s) módulo(s) de aprofundamento e gere mais arquivos de *run*.
6. Escreva tudo no relatório SBC. ✍️